In [ ]:
# Fix: Kaggle's default NumPy 1.26.4 is CORRECT - prevent upgrades to 2.x
import sys
print("Step 1: Record Kaggle's default NumPy version...")
import numpy as np
default_numpy = np.__version__
print(f"   Default NumPy: {default_numpy}")

print("\nStep 2: Install ultralytics...")
import subprocess
subprocess.run(["pip", "install", "-q", "ultralytics"], check=True)

print("\nStep 3: LOCK NumPy back to Kaggle's default...")
subprocess.run(["pip", "install", "-q", "--force-reinstall", f"numpy=={default_numpy}"], check=True)

print("\nInstallation complete!")
import numpy as np
from ultralytics import YOLO
print(f"   NumPy: {np.__version__}")
print("   OK Ultralytics imported successfully")

In [ ]:
import json
import os
import sys
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO

# Environment detection
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

# KAGGLE DATASET SETUP
if os.path.exists("/kaggle/input/wound-seg-code-v1"):
    sys.path.insert(0, "/kaggle/input/wound-seg-code-v1")
    print("[INFO] Code dataset added to Python path")
    from evaluation.metrics import calculate_segmentation_metrics, calculate_boundary_f1, calculate_bootstrap_ci
    from pipeline.eda import EDA_CONFIG, Visualizer
    from data.yolo_config import create_yolo_config, validate_yolo_config
    from evaluation.yolo_utils import yolo_result_to_metrics_input
else:
    sys.path.insert(0, str(PROJECT_ROOT))
    print("[INFO] Using local src directory")
    from src.evaluation.metrics import calculate_segmentation_metrics, calculate_boundary_f1, calculate_bootstrap_ci
    from src.pipeline.eda import EDA_CONFIG, Visualizer
    from src.data.yolo_config import create_yolo_config, validate_yolo_config
    from src.evaluation.yolo_utils import yolo_result_to_metrics_input

print("OK All modules imported successfully")

In [ ]:
# Configure paths for Kaggle
if os.path.exists("/kaggle/input"):
    IN_KAGGLE = True
    BASE_INPUT_PATH = Path("/kaggle/input")
    OUTPUT_PATH = Path("/kaggle/working")
    DATASET_PATH = BASE_INPUT_PATH / "tiny-wound-seg-smoke-test"
    CODE_PATH = BASE_INPUT_PATH / "wound-seg-code-v1"
else:
    IN_KAGGLE = False
    BASE_INPUT_PATH = PROJECT_ROOT / "data"
    OUTPUT_PATH = PROJECT_ROOT / "outputs"
    DATASET_PATH = BASE_INPUT_PATH / "TINY_TEST"
    CODE_PATH = PROJECT_ROOT / "src"

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Local'}")
print(f"Dataset path: {DATASET_PATH}")
print(f"Output path: {OUTPUT_PATH}")

# Verify dataset exists
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {DATASET_PATH}")

print("OK Environment configured")

In [ ]:
# Use baseline-outputs pre-converted YOLO dataset (not TINY)
YOLO_DATASET_PATH = BASE_INPUT_PATH / "baseline-outputs/data_wound_seg"

CONFIG_PATH = OUTPUT_PATH / "wound_dataset_smoke.yaml"
config_result = create_yolo_config(
    dataset_root=YOLO_DATASET_PATH,
    train_dir="train",  # YOLO format: train/images + train/labels
    val_dir="test",  # YOLO format: test/images + test/labels
    test_dir="test",
    output_path=CONFIG_PATH
)

if not config_result["success"]:
    raise RuntimeError(f"YOLO config creation failed: {config_result['error']}")

print(f"OK YOLO config created: {CONFIG_PATH}")

# Validate config
validation_result = validate_yolo_config(CONFIG_PATH)
if not validation_result["success"]:
    raise RuntimeError(f"YOLO config validation failed: {validation_result['error']}")

print("OK YOLO config validated")

## Training Setup

**Intervention**: Label Smoothing

This smoke test validates the training pipeline with 1 epoch on TINY dataset (2-5 images).


In [ ]:
# Single-seed smoke test (seed 42 only)
TRAINING_OUTPUT = OUTPUT_PATH / "yolo_training"
seed = 42

# Label Smoothing Training Parameters
model = YOLO("yolov8n-seg.pt")
model.train(
    data=str(CONFIG_PATH),
    epochs=1,  # Smoke test: 1 epoch only
    imgsz=512,
    batch=4,  # TINY dataset: small batch
    lr0=0.015,
    workers=2,
    amp=False,
    project=str(TRAINING_OUTPUT),
    name=f"seed_{seed}",
    exist_ok=True,
    device='cpu',  # CPU smoke test
    verbose=True,
    # INTERVENTION: Label Smoothing
    label_smoothing=0.1,  # Soft targets (0.1, 0.9) instead of (0, 1)
    mixup=0.0,  # Disabled (unsafe for medical imaging)
)

print(f"OK Training completed for seed {seed}")

# Verify weights exist
weights_path = TRAINING_OUTPUT / f"seed_{seed}" / "weights" / "best.pt"
if not weights_path.exists():
    raise FileNotFoundError(f"Weights not found at {weights_path}")

print(f"OK Weights saved: {weights_path}")


In [ ]:
# Evaluation on baseline-outputs test set (use subset for smoke test)
TEST_IMAGES_DIR = YOLO_DATASET_PATH / "test/images"
TEST_MASKS_DIR = YOLO_DATASET_PATH / "test_masks"

# Load trained model
model = YOLO(str(weights_path))

# Evaluate on first 3 test images (smoke test)
test_images = sorted(TEST_IMAGES_DIR.glob("*.png"))[:3]

results = []
for img_path in test_images:
    image_id = img_path.stem
    mask_path = TEST_MASKS_DIR / f"{image_id}.png"

    if not mask_path.exists():
        print(f"WARNING  Skipping {image_id} - no ground truth mask")
        continue

    # Load image
    img_rgb = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_BGR2RGB)

    # Standard evaluation (no TTA)
    yolo_results = model.predict(img_rgb, verbose=False)

    # Convert to metrics input
    mask_path_str = str(mask_path)
    conversion_result = yolo_result_to_metrics_input(
        yolo_results[0],
        mask_path_str,
        image_id
    )

    if not conversion_result["success"]:
        raise RuntimeError(f"Conversion failed: {conversion_result['error']}")

    data = conversion_result["data"]
    metrics_result = calculate_segmentation_metrics(
        data["pred_mask"],
        data["gt_mask"],
        image_id
    )
    results.append(metrics_result)

    print(f"OK Evaluated {image_id}: Dice={metrics_result['dice']:.4f}")

print(f"OK Evaluation completed on {len(results)} images")

# Keep last result for schema validation
if results:
    metrics_result = results[-1]

In [ ]:
# Validate metrics schema (lowercase column names)
expected_keys = {'tp', 'fp', 'tn', 'fn', 'dice', 'iou', 'precision', 'recall'}
actual_keys = set(metrics_result.keys())

missing_keys = expected_keys - actual_keys
extra_keys = actual_keys - expected_keys

if missing_keys:
    raise AssertionError(f"Missing keys in metrics: {missing_keys}")

if extra_keys:
    print(f"WARNING  Extra keys in metrics: {extra_keys} (not an error, just FYI)")

print("OK Schema validation passed")
print(f"  Metrics keys: {sorted(actual_keys)}")

In [ ]:
# Verify output files created
smoke_test_report = {
    "ablation": "6",
    "intervention": "Label Smoothing",
    "validation_checks": {
        "numpy_locking": {"passed": True},
        "imports": {"passed": True},
        "yolo_config": {"passed": True},
        "training": {"passed": True, "epochs": 1},
        "evaluation": {"passed": True, "images_tested": 1},
        "schema_validation": {"passed": True},
    },
    "metrics_sample": {
        "dice": float(metrics_result['dice']),
        "iou": float(metrics_result['iou']),
        "recall": float(metrics_result['recall']),
        "precision": float(metrics_result['precision']),
    }
}

report_path = OUTPUT_PATH / "smoke_test_report.json"
with open(report_path, 'w') as f:
    json.dump(smoke_test_report, f, indent=2)

print(f"OK Smoke test report saved: {report_path}")
print("\nALL SMOKE TEST VALIDATIONS PASSED!")
